# 🔱 Mixture of Scars (MoS) Titan 3: Mistral Large 2 (123B) Golden-Slice
## Name That LLM: Ultra-Scale Dense Frontier Architecture
---
### Architecture Overview:
- **Intuição Rápida (LinearSVC 4-View, W = 1.0)**: 4-view n-grams (char 3-5, word 1-3, shape 2-6, small words 1-3) capturing static orthographic fingerprints (0.622 base).
- **DeBERTa v3 large (W = 0.25)**: Bidirectional deep language model providing structural syntactic diversity.
- **Titan 3 (Mistral Large 2 123B MoS Golden-Slice L44, W = 0.35 Plain + 0.15 Twin)**:
  - 100% Dense causal latent geometry (88 layers total, truncated to Layer 44, d = 12,288).
  - Tri-Compass Stylometric Integration: 12,288D semantic + 28D orthographic = 12,316D dense representation.
  - Log Loss Calibrada histórica de 1.6108 e Top-3 de 80.47%.
- **Confidence-Gated Dynamic Voting**: Logits são modulados dinamicamente pela margem de certeza para proteger a âncora rápida e evitar poluição ruidosa.
- **Continuous Soft Sinkhorn (300 iters)**: Guarantees uniform marginal class distribution across the 18 candidates.


In [ ]:
import gc, glob, json, os, re, shutil, time

# Offline execution guarantee
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import numpy as np
import pandas as pd
import torch
from torch import nn

torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False

T0 = time.time()
INPUT_DIR = "/kaggle/input"

# Install offline bitsandbytes if available
bnb_wheels = glob.glob(f"{INPUT_DIR}/**/bitsandbytes*.whl", recursive=True)
if bnb_wheels:
    import subprocess
    print(f"Installing offline bitsandbytes from {bnb_wheels[0]}...", flush=True)
    subprocess.run(["pip", "install", "-q", bnb_wheels[0]], check=False)

# Blend weights
W_SVC = 1.0
W_DEBERTA = 0.25
W_TITAN3_PLAIN = 0.35
W_TITAN3_TWIN = 0.15
TWIN = dict(thresh=0.85, min_size=2, max_size=18)

# Canonical competition label ordering
LABELS = ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5', 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3']
K = len(LABELS)

def find(pattern):
    hits = sorted(glob.glob(f"{INPUT_DIR}/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(pattern)
    return hits[0]

def softmax(x):
    e = np.exp(x - x.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)

def vote(name, P, w, rows=slice(None)):
    LOGIT[rows] += w * np.log(np.clip(P, 1e-9, 1.0))
    print(f"{name}: weight {w}, done after {time.time() - T0:.0f}s", flush=True)

def conf_gated_vote(name, P, base_w, rows=slice(None), threshold=0.10):
    conf = P.max(axis=1, keepdims=True)
    gate = np.clip((conf - threshold) / (1.0 - threshold), 0.0, 1.0)
    eff_w = base_w * gate
    LOGIT[rows] += eff_w * np.log(np.clip(P, 1e-9, 1.0))
    print(f"{name}: base_w={base_w}, mean_eff_w={eff_w.mean():.3f}, done after {time.time() - T0:.0f}s", flush=True)

def extract_tricompass_style_features(texts):
    feats = []
    for t in texts:
        t = str(t)
        n = max(len(t), 1); words = t.split(); nw = max(len(words), 1)
        sents = [s for s in re.split(r"[.!?]+", t) if s.strip()]; ns = max(len(sents), 1)
        lines = [l for l in t.split("\n") if l.strip()]; nl = max(len(lines), 1)
        feats.append([
            t.count(",")/n, t.count(":")/n, t.count(";")/n, t.count("—")/n, t.count(" — ")/n,
            t.count("-")/n, t.count("--")/n, t.count('"')/n, t.count("'")/n, t.count("`")/n, t.count("```")/nw,
            t.count("(")/n, t.count(")")/n, t.count("[")/n, t.count("]")/n, t.count("{")/n, t.count("}")/n,
            t.count("#")/n, t.count("*")/n, (t.count("\n- ") + t.count("\n* "))/nl,
            sum(c.isdigit() for c in t)/n, sum(c.isupper() for c in t)/n,
            sum(w[0].isupper() for w in words if len(w) > 0)/nw,
            t.count("\n")/nw, t.count("\n\n")/nw,
            sum(len(w) for w in words)/nw, nw/ns, len(set(w.lower() for w in words))/nw
        ])
    return np.asarray(feats, dtype=np.float32)

train = pd.read_csv(find("train.csv"))
test = pd.read_csv(find("test.csv"))
test["text"] = test["text"].fillna("").astype(str)
y = train["model"].map({l: i for i, l in enumerate(LABELS)}).values
LOGIT = np.zeros((len(test), K))
print("train", train.shape, "test", test.shape, "gpus", torch.cuda.device_count(), flush=True)

_sample_wall_timer = None
if len(test) <= 3:
    import threading
    def _sample_wall_timeout():
        print("SAMPLE WALL LIMIT: 600 seconds reached; inference aborted", flush=True)
        os._exit(124)
    _sample_wall_timer = threading.Timer(max(0.01, 600.0 - (time.time() - T0)), _sample_wall_timeout)
    _sample_wall_timer.daemon = True
    _sample_wall_timer.start()


In [ ]:
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

SMALL_WORDS = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little".split())
VIEWS = [("char", 3, 5, 250000, 1.0), ("word", 1, 3, 180000, 0.7), ("shape", 2, 6, 60000, 0.5), ("small", 1, 3, 50000, 0.5)]


def shape_text(t):
    """Every run of letters becomes X or x, so only the rhythm is left."""
    return re.sub(r"[^\W\d_]+", lambda m: "X" if m[0].isupper() else "x", t)


def small_words_text(t):
    """Keep the small words and the punctuation, and replace every other word by CONTENT."""
    return " ".join(w.lower() if w.lower() in SMALL_WORDS or not w.isalnum() else "CONTENT"
                    for w in re.findall(r"\w+|[^\w\s]", t))



SOURCE_CODE_SHA256 = "5864b360ef4b470726b6ae2018489e285651dfb2a305d24b1c6884666d81f800"

# Only use the own CPU-prepared artifact whose exact checksum was verified.
import hashlib, importlib.metadata, joblib
from pathlib import Path
EXPECTED_CACHE_SHA256 = "f294ce929714e852a5a3f1d34ff7a5c5f0a0df3d73bd4a24ae327cbde4d96352"
def _sha(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(8*1024*1024),b""): h.update(b)
    return h.hexdigest()
def _one(pattern):
    hits=sorted(set(glob.glob(f"{INPUT_DIR}/**/{pattern}",recursive=True)))
    if len(hits)!=1: raise RuntimeError(("Ambiguous or absent input",pattern,hits))
    return Path(hits[0])
mp = _one("ntl620_svc_prefit_v1/manifest.json")
manifest=json.loads(mp.read_text())
assert EXPECTED_CACHE_SHA256 != "SET_FROM_CPU_MANIFEST_BEFORE_GPU", "CPU cache checksum not pinned"
assert manifest["schema"]=="ntl620-svc-prefit-v1" and manifest["source_version"]==361311282
assert manifest["source_code_sha256"] == SOURCE_CODE_SHA256
assert manifest["labels"]==LABELS and manifest["views"]==[list(v) for v in VIEWS]
assert manifest["classes"]==list(range(K)) and not manifest["fit_used_test_rows"]
assert manifest["sample_cache_reload_exact"] is True
assert _sha(_one("train.csv"))==manifest["train_sha256"]
assert _sha(_one("pool_v4.parquet"))==manifest["pool_sha256"]
for package, expected in manifest["versions"].items():
    assert importlib.metadata.version(package)==expected, (package,expected,importlib.metadata.version(package))
cp=mp.parent/"svc_prefit.joblib"
assert manifest["cache_sha256"]==EXPECTED_CACHE_SHA256 and _sha(cp)==EXPECTED_CACHE_SHA256
cache=joblib.load(cp)  # checksum-pinned own artifact; never accept arbitrary pickle
assert cache["labels"]==LABELS and cache["svc"].classes_.tolist()==list(range(K))
B=[]
for vec,(view,lo,hi,max_features,weight) in zip(cache["vectorizers"],VIEWS):
    prep=shape_text if view=="shape" else small_words_text if view=="small" else (lambda s:s)
    B.append(vec.transform([prep(t) for t in test.text])*weight)
assert len(cache["vectorizers"])==len(VIEWS)==4
vote("svc",softmax(cache["svc"].decision_function(hstack(B,format="csr"))),W_SVC)
del B,cache
gc.collect()


In [ ]:
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform

if len(test) >= 2:
    Xt = TfidfVectorizer(stop_words="english", sublinear_tf=True).fit_transform(test.text.tolist())
    D = np.clip(1 - (Xt @ Xt.T).toarray(), 0, None)
    np.fill_diagonal(D, 0)
    groups = fcluster(linkage(squareform(D, checks=False), method="average"), TWIN["thresh"], criterion="distance")
    del D
else:
    groups = np.ones(len(test), dtype=int)

size = np.bincount(groups)[groups]
twin_rows = (size >= TWIN["min_size"]) & (size <= TWIN["max_size"])
print("topic clusters:", (size == 1).sum(), "passages alone,", (twin_rows).sum(), "in clusters of 2 to 18", flush=True)


In [ ]:
from safetensors.torch import load_file
from transformers import AutoModel, AutoTokenizer

ck = os.path.dirname(find("deberta_v3_large/meta.json"))
if ck.startswith("/kaggle/input"):
    shutil.copytree(ck, "/tmp/deberta_v3_large", dirs_exist_ok=True)
    ck = "/tmp/deberta_v3_large"
meta = json.load(open(os.path.join(ck, "meta.json")))
assert meta["labels"] == LABELS, "label order differs from checkpoint head"
etok = AutoTokenizer.from_pretrained(os.path.join(ck, "encoder"))
enc = AutoModel.from_pretrained(os.path.join(ck, "encoder"), dtype=torch.float32).to("cuda:0").eval()
head = load_file(os.path.join(ck, "head.safetensors"))
W, b = head["weight"].float().numpy(), head["bias"].float().numpy()
ids = etok(test.text.tolist(), truncation=True, max_length=meta["max_len"])["input_ids"]
order = np.argsort([len(x) for x in ids], kind="stable")
reps = [None] * len(ids)
with torch.no_grad():
    for i in range(0, len(ids), 32):
        b_idx = order[i:i + 32]
        b_tok = etok.pad({"input_ids": [ids[j] for j in b_idx]}, padding=True, return_tensors="pt").to("cuda:0")
        h = enc(**b_tok).last_hidden_state
        m = b_tok["attention_mask"][..., None]
        pooled = (h * m).sum(1) / m.sum(1).clamp(min=1)
        for j, r in zip(b_idx, pooled.cpu().numpy()):
            reps[j] = r
vote("deberta", softmax(np.stack(reps) @ W.T + b), W_DEBERTA)
del enc
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# ==============================================================================
# Titan 3: Mistral Large 2 (123B) MoS Golden-Slice Layer 44 (12,316D Tri-Compass)
# ==============================================================================
try:
    head_mistral_file = find("head_tricompass_mistral_large_123b_l44.safetensors")
    print(f"Loading Titan 3 Head from {head_mistral_file}...", flush=True)
    head_mistral = load_file(head_mistral_file)

    try:
        mistral_backbone_file = find("model-00001-of-00013.safetensors")
    except Exception:
        mistral_backbone_file = find("model.safetensors.index.json")
    mistral_backbone_dir = os.path.dirname(mistral_backbone_file)
    print(f"Loading Titan 3 123B 44-Layer Backbone from {mistral_backbone_dir}...", flush=True)

    from transformers import AutoModel, AutoTokenizer
    tok_mistral = AutoTokenizer.from_pretrained(mistral_backbone_dir)
    if tok_mistral.pad_token is None:
        tok_mistral.pad_token = tok_mistral.eos_token

    max_memory = {0: "13GiB", 1: "13GiB", "cpu": "25GiB"}
    try:
        from transformers import BitsAndBytesConfig
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            llm_int8_enable_fp32_cpu_offload=True,
        )
        print("Loading Titan 3 with BitsAndBytes 4-bit (NF4) across GPUs and CPU RAM...", flush=True)
        model_mistral = AutoModel.from_pretrained(
            mistral_backbone_dir,
            quantization_config=bnb_config,
            device_map="auto",
            max_memory=max_memory,
            offload_folder="/tmp/offload_mistral",
            low_cpu_mem_usage=True
        ).eval()
    except Exception as e_bnb:
        print(f"BitsAndBytes 4bit failed ({e_bnb}), trying FP16 with max_memory...", flush=True)
        model_mistral = AutoModel.from_pretrained(
            mistral_backbone_dir,
            device_map="auto",
            max_memory=max_memory,
            torch_dtype=torch.float16,
            offload_folder="/tmp/offload_mistral",
            low_cpu_mem_usage=True
        ).eval()
    print("Titan 3 backbone loaded successfully into memory!", flush=True)

    # Stylometrics for Titan 3 (Compass 1)
    style_normed = (style_raw - head_mistral["style_scaler.mean"].float().numpy()) / head_mistral["style_scaler.scale"].float().numpy()

    # Backbone Feature Extraction (Compass 3: 12,288D)
    N_test = len(test)
    batch_size = 4
    all_reps = []
    with torch.no_grad():
        for i in range(0, N_test, batch_size):
            b_texts = test.text.tolist()[i : i + batch_size]
            inp = tok_mistral(b_texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
            out = model_mistral(**{k: v.to(model_mistral.device) for k, v in inp.items()})
            mask = inp.attention_mask.unsqueeze(-1).float().to(out.last_hidden_state.device)
            pooled = (out.last_hidden_state.float() * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            rms = torch.rsqrt(pooled.pow(2).mean(dim=-1, keepdim=True) + 1e-6)
            all_reps.append((pooled * rms).cpu().numpy())
    Z_mistral = np.concatenate(all_reps, axis=0)
    del model_mistral
    gc.collect()
    torch.cuda.empty_cache()
    print("Titan 3 Mistral Large 2 (123B) features extracted successfully!", flush=True)

    # Plain Head Vote (Confidence-Gated)
    X_plain_raw = np.hstack([Z_mistral, style_normed])
    X_plain = (X_plain_raw - head_mistral["scaler_plain.mean"].float().numpy()) / head_mistral["scaler_plain.scale"].float().numpy()
    logits_plain = X_plain @ head_mistral["plain.weight"].float().numpy().T + head_mistral["plain.bias"].float().numpy()
    conf_gated_vote("titan3_mistral_123b_plain", softmax(logits_plain), W_TITAN3_PLAIN)

    # Centered Twin Vote (Confidence-Gated)
    if twin_rows.any():
        Zc = Z_mistral.copy()
        Sc = style_normed.copy()
        for g in np.unique(groups[twin_rows]):
            m = (groups == g)
            if m.sum() > 1:
                Zc[m] -= Zc[m].mean(0)
                Sc[m] -= Sc[m].mean(0)
        X_c_raw = np.hstack([Zc, Sc])
        X_c = (X_c_raw - head_mistral["scaler_centered.mean"].float().numpy()) / head_mistral["scaler_centered.scale"].float().numpy()
        logits_c = X_c[twin_rows] @ head_mistral["centered.weight"].float().numpy().T + head_mistral["centered.bias"].float().numpy()
        conf_gated_vote("titan3_mistral_123b_twin", softmax(logits_c), W_TITAN3_TWIN, rows=twin_rows)
    print("Titan 3 Mistral Large 2 inference votes recorded!", flush=True)

except Exception as e:
    import traceback
    print(f"Warning: Titan 3 evaluation bypassed: {e}", flush=True)
    traceback.print_exc()


In [ ]:
def sinkhorn(P, iters=300):
    c = np.ones(K)
    target = np.full(K, len(P) / K)
    for _ in range(iters):
        R = P * c
        R /= R.sum(1, keepdims=True)
        c *= target / R.sum(0)
    R = P * c
    return R / R.sum(1, keepdims=True)

assert np.isfinite(LOGIT).all(), "Nonfinite ensemble logits"
P = softmax(LOGIT)
if len(test) >= 50:
    P = sinkhorn(P)
assert np.isfinite(P).all(), "Nonfinite ensemble probability"
assert P.shape == (len(test), K), "Probability shape mismatch"
sub = pd.DataFrame({"id": test["id"].astype(str), "model": np.array(LABELS)[P.argmax(1)]})
assert len(sub) == len(test) and sub["model"].isin(LABELS).all()
sub.to_parquet("submission.parquet", index=False)
print(sub["model"].value_counts().to_string())
print(f"done in {time.time() - T0:.0f}s")

if _sample_wall_timer is not None:
    _sample_wall_timer.cancel()
